# **`Inatel - C318 (Tópicos Especiais II) - 2026/1`**

# <font color='green'>**Atividade 07: HPO e AutoML**</font>

## <font color='#2D9CDB'>**LEIA ATENTAMENTE AS INSTRUÇÕES A SEGUIR**</font>
- Importe este notebook no [Google Colab](https://colab.research.google.com/) para resolver os exercícios;
- Consulte a apostila disponibilizada pelo professor para se familiarizar com os conceitos;
- Utilize os recursos disponíveis na Internet (documentações e artigos científicos) para complementar seus estudos;
- <font color='red'>**Uso consciente de Inteligência Artificial (LLMs):**</font>
  - O uso de assistentes (como Gemini, ChatGPT, Claude) é permitido, mas exige responsabilidade técnica:
    - Em vez de pedir a solução completa, peça para a IA explicar conceitos, sugerir abordagens ou ajudar a depurar erros de código;
    - Você é o responsável por cada linha de código entregue. Não insira no notebook implementações que você não compreende integralmente ou não saberia explicar;
    - Modelos de linguagem podem "alucinar" funções ou sugerir métodos obsoletos de bibliotecas em Python. Sempre teste e verifique a documentação oficial;
    - Quando utilizar a IA para gerar ou refatorar blocos lógicos complexos, indique isso através de comentários no próprio código;
- <font color='red'>**NÃO**</font> remova as células de Código já presentes neste notebook;
- <font color='red'>**NÃO**</font> modifique as células de Markdown (em <font color='green'>verde</font> ou <font color='#2D9CDB'>azul</font>) presentes neste notebook;
- Após cada questão, há uma célula para você implementar e responder a questão;
- É permitido adicionar mais células (de código ou markdown) antes da próxima pergunta;
- Caso precise utilizar bibliotecas que não estão instaladas nativamente no Colab, inclua uma célula de código com o comando de instalação (ex: `!pip install nome_da_biblioteca`);
- <font color='red'>**Renomeie o termo `_Enunciado` para `_seu_numero_de_matricula` no nome do arquivo (exemplo: `C318_2026_1_Atividade_07_12345.ipynb`)**</font>;
- <font color='magenta'>**Faça download do notebook com a resolução no Google Colab, mantendo a saída de todas as células, e anexe-o à tarefa do Teams.**</font>

# <font color='green'><u><b>Preparação</b></u></font>

In [28]:
!pip install numpy pandas matplotlib seaborn scikit-learn optuna

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from sklearn.datasets import load_wine
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.datasets import load_breast_cancer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

np.random.seed(42)

# <font color='green'><u><b>Parte 1 - Otimização de Hiperparâmetros (HPO)</b></u></font>

### <font color='#2D9CDB'>Q2) Escolha um problema de classificação disponível no módulo <a href="https://scikit-learn.org/stable/api/sklearn.datasets.html" target="blank">sklearn.datasets</a> ou no repositório <a href="https://archive.ics.uci.edu/datasets/?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true" target="blank">UCI Machine Learning Repository</a>.</font>
<ul>
    <font color='#2D9CDB'><li>Utilizando um modelo de regressão diferente daquele empregado nos exemplos da apostila, realize a otimização de hiperparâmetros com Optuna.</li></font>
    <font color='#2D9CDB'><li>Defina um espaço de busca contendo pelo menos três hiperparâmetros, utilize validação cruzada para avaliação e execute pelo menos 30 trials.</li></font>
    <font color='#2D9CDB'><li>Apresente os melhores hiperparâmetros encontrados, a métrica de classificação utilizada e o desempenho obtido pelo modelo otimizado.</li></font>
    <font color='#2D9CDB'><li>Em seguida, compare os resultados com aqueles obtidos utilizando a configuração padrão do algoritmo e discuta os ganhos observados.</li></font>
</ul>
<font color='2D9CDB'>Não é permitido utilizar o dataset Digits nem o mesmo modelo apresentado nos exemplos da apostila.</font>

In [30]:
dados = load_wine()

X = dados.data
y = dados.target

print("Dataset:", dados.DESCR.split("\n")[0])
print("Quantidade de amostras:", X.shape[0])
print("Número de atributos:", X.shape[1])
print("Classes:", dados.target_names)

Dataset: .. _wine_dataset:
Quantidade de amostras: 178
Número de atributos: 13
Classes: ['class_0' 'class_1' 'class_2']


In [31]:
dados = load_wine()

X = dados.data
y = dados.target

print("Dataset:", dados.DESCR.split("\n")[0])
print("Quantidade de amostras:", X.shape[0])
print("Número de atributos:", X.shape[1])
print("Classes:", dados.target_names)

Dataset: .. _wine_dataset:
Quantidade de amostras: 178
Número de atributos: 13
Classes: ['class_0' 'class_1' 'class_2']


In [32]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 1, 30)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_int("p", 1, 3)
    leaf_size = trial.suggest_int("leaf_size", 10, 60)

    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            p=p,
            leaf_size=leaf_size
        ))
    ])

    scores = cross_val_score(
        modelo,
        X,
        y,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    return scores.mean()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=30)

print("Melhores hiperparâmetros encontrados:")
print(study.best_params)

print(f"\nMelhor F1-macro médio na validação cruzada: {study.best_value:.4f}")

[I 2026-06-28 21:21:36,363] A new study created in memory with name: no-name-8bd444f6-9b5a-4ba0-a0e1-34c065896d48
[I 2026-06-28 21:21:46,031] Trial 0 finished with value: 0.9675919142585808 and parameters: {'n_neighbors': 12, 'weights': 'uniform', 'p': 2, 'leaf_size': 17}. Best is trial 0 with value: 0.9675919142585808.
[I 2026-06-28 21:21:46,389] Trial 1 finished with value: 0.9721091923245584 and parameters: {'n_neighbors': 5, 'weights': 'distance', 'p': 2, 'leaf_size': 46}. Best is trial 1 with value: 0.9721091923245584.
[I 2026-06-28 21:21:46,529] Trial 2 finished with value: 0.9777192839648979 and parameters: {'n_neighbors': 1, 'weights': 'uniform', 'p': 1, 'leaf_size': 19}. Best is trial 2 with value: 0.9777192839648979.
[I 2026-06-28 21:21:46,682] Trial 3 finished with value: 0.9609758338276185 and parameters: {'n_neighbors': 6, 'weights': 'distance', 'p': 2, 'leaf_size': 24}. Best is trial 2 with value: 0.9777192839648979.
[I 2026-06-28 21:21:46,909] Trial 4 finished with value

Melhores hiperparâmetros encontrados:
{'n_neighbors': 17, 'weights': 'distance', 'p': 1, 'leaf_size': 39}

Melhor F1-macro médio na validação cruzada: 0.9789


In [33]:
modelo_padrao = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

scores_padrao = cross_val_score(
    modelo_padrao,
    X,
    y,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

f1_padrao = scores_padrao.mean()


modelo_otimizado = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(**study.best_params))
])

scores_otimizado = cross_val_score(
    modelo_otimizado,
    X,
    y,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

f1_otimizado = scores_otimizado.mean()

ganho_absoluto = f1_otimizado - f1_padrao
ganho_percentual = (ganho_absoluto / f1_padrao) * 100

resultados = pd.DataFrame({
    "Modelo": ["KNN padrão", "KNN otimizado"],
    "F1-macro médio": [f1_padrao, f1_otimizado],
    "Desvio padrão": [scores_padrao.std(), scores_otimizado.std()]
})

display(resultados)

print(f"F1-macro com KNN padrão: {f1_padrao:.4f}")
print(f"F1-macro com KNN otimizado: {f1_otimizado:.4f}")
print(f"Ganho absoluto: {ganho_absoluto:.4f}")
print(f"Ganho percentual: {ganho_percentual:.2f}%")

,Modelo,F1-macro médio,Desvio padrão
0,KNN padrão,0.972109,0.017909
1,KNN otimizado,0.978949,0.010571


F1-macro com KNN padrão: 0.9721
F1-macro com KNN otimizado: 0.9789
Ganho absoluto: 0.0068
Ganho percentual: 0.70%


Foi utilizado o Wine Dataset, disponível no sklearn.datasets, em um problema de classificação multiclasse. O modelo escolhido foi o KNeighborsClassifier, diferente do modelo usado no exemplo da apostila, e o dataset Digits não foi utilizado nas tarefas.

A otimização foi feita com Optuna, usando 30 trials e validação cruzada estratificada com 5 folds. A métrica utilizada foi o F1-macro. O espaço de busca incluiu os hiperparâmetros n_neighbors, weights, p e leaf_size.

Os melhores hiperparâmetros encontrados foram: n_neighbors = 17, weights = distance, p = 1 e leaf_size = 39. O KNN padrão obteve F1-macro médio de 0.9721, enquanto o KNN otimizado obteve 0.9789.

Assim, houve um ganho absoluto de 0.0068, equivalente a aproximadamente 0.70%. O ganho foi pequeno, mas positivo, indicando que a otimização melhorou levemente o desempenho e deixou o modelo mais estável.


### <font color='#2D9CDB'>Q2) Escolha um problema de regressão disponível no módulo <a href="https://scikit-learn.org/stable/api/sklearn.datasets.html" target="blank">sklearn.datasets</a> ou no repositório <a href="https://archive.ics.uci.edu/datasets/?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true" target="blank">UCI Machine Learning Repository</a>.</font>
<ul>
    <font color='#2D9CDB'><li>Utilizando um modelo de classificação diferente daquele empregado nos exemplos da apostila, realize a otimização de hiperparâmetros com Optuna.</li></font>
    <font color='#2D9CDB'><li>Defina um espaço de busca contendo pelo menos três hiperparâmetros, utilize validação cruzada para avaliação e execute pelo menos 30 trials.</li></font>
    <font color='#2D9CDB'><li>Apresente os melhores hiperparâmetros encontrados, a métrica de classificação utilizada e o desempenho obtido pelo modelo otimizado.</li></font>
    <font color='#2D9CDB'><li>Em seguida, compare os resultados com aqueles obtidos utilizando a configuração padrão do algoritmo e discuta os ganhos observados.</li></font>
</ul>
<font color='2D9CDB'>Não é permitido utilizar o mesmo dataset nem o mesmo modelo apresentado nos exemplos da apostila.</font>

In [34]:
dados = load_diabetes()

X = dados.data
y = dados.target

print("Dataset:", dados.DESCR.split("\n")[0])
print("Quantidade de amostras:", X.shape[0])
print("Número de atributos:", X.shape[1])
print("Tipo do alvo: valor numérico contínuo")

Dataset: .. _diabetes_dataset:
Quantidade de amostras: 442
Número de atributos: 10
Tipo do alvo: valor numérico contínuo


In [35]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
    C = trial.suggest_float("C", 0.01, 100, log=True)
    epsilon = trial.suggest_float("epsilon", 0.01, 5, log=True)
    gamma = trial.suggest_float("gamma", 0.0001, 1, log=True)

    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(
            kernel=kernel,
            C=C,
            epsilon=epsilon,
            gamma=gamma
        ))
    ])

    scores = cross_val_score(
        modelo,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )

    return scores.mean()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=30)

print("Melhores hiperparâmetros encontrados:")
print(study.best_params)

print(f"\nMelhor RMSE médio na validação cruzada: {-study.best_value:.4f}")

[I 2026-06-28 21:21:53,299] A new study created in memory with name: no-name-ded289df-cd4d-4c81-b579-7d2c30c29e23
[I 2026-06-28 21:21:53,692] Trial 0 finished with value: -75.39838057080752 and parameters: {'kernel': 'rbf', 'C': 8.471801418819979, 'epsilon': 0.4128205343826223, 'gamma': 0.00042079886696066364}. Best is trial 0 with value: -75.39838057080752.
[I 2026-06-28 21:21:54,332] Trial 1 finished with value: -55.51384131490097 and parameters: {'kernel': 'linear', 'C': 29.154431891537552, 'epsilon': 0.4191711516695202, 'gamma': 0.06796578090758151}. Best is trial 1 with value: -55.51384131490097.
[I 2026-06-28 21:21:54,796] Trial 2 finished with value: -71.15891256228564 and parameters: {'kernel': 'rbf', 'C': 21.368329072358772, 'epsilon': 0.037419406111184966, 'gamma': 0.000533703276260396}. Best is trial 1 with value: -55.51384131490097.
[I 2026-06-28 21:21:55,332] Trial 3 finished with value: -76.69789246662621 and parameters: {'kernel': 'rbf', 'C': 1.256104370001356, 'epsilon'

Melhores hiperparâmetros encontrados:
{'kernel': 'linear', 'C': 0.7936593801664463, 'epsilon': 1.9074559309852333, 'gamma': 0.4079703441566138}

Melhor RMSE médio na validação cruzada: 54.8832


In [36]:
modelo_padrao = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

scores_padrao = cross_val_score(
    modelo_padrao,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

rmse_padrao = -scores_padrao.mean()


modelo_otimizado = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(**study.best_params))
])

scores_otimizado = cross_val_score(
    modelo_otimizado,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

rmse_otimizado = -scores_otimizado.mean()

ganho_absoluto = rmse_padrao - rmse_otimizado
ganho_percentual = (ganho_absoluto / rmse_padrao) * 100

resultados = pd.DataFrame({
    "Modelo": ["SVR padrão", "SVR otimizado"],
    "RMSE médio": [rmse_padrao, rmse_otimizado],
    "Desvio padrão": [scores_padrao.std(), scores_otimizado.std()]
})

display(resultados)

print(f"RMSE com SVR padrão: {rmse_padrao:.4f}")
print(f"RMSE com SVR otimizado: {rmse_otimizado:.4f}")
print(f"Ganho absoluto: {ganho_absoluto:.4f}")
print(f"Ganho percentual: {ganho_percentual:.2f}%")

,Modelo,RMSE médio,Desvio padrão
0,SVR padrão,70.532507,3.841769
1,SVR otimizado,54.883197,1.923309


RMSE com SVR padrão: 70.5325
RMSE com SVR otimizado: 54.8832
Ganho absoluto: 15.6493
Ganho percentual: 22.19%


Foi utilizado o dataset Diabetes, disponível no sklearn.datasets, em um problema de regressão. O modelo escolhido foi o SVR, diferente do modelo utilizado no exemplo da apostila.

A otimização foi realizada com Optuna, usando 30 trials e validação cruzada com 5 folds. A métrica utilizada foi o RMSE, que mede o erro médio das previsões, sendo melhor quando apresenta valores menores. O espaço de busca incluiu os hiperparâmetros kernel, C, epsilon e gamma.

Os melhores hiperparâmetros encontrados foram: kernel = linear, C = 0.7937, epsilon = 1.9075 e gamma = 0.4080. O SVR padrão obteve RMSE médio de 70.5325, enquanto o SVR otimizado obteve RMSE médio de 54.8832.

Assim, houve uma redução absoluta de 15.6493 no RMSE, equivalente a uma melhoria de aproximadamente 22.19%. Além disso, o desvio padrão também diminuiu, indicando que o modelo otimizado apresentou resultados mais estáveis durante a validação cruzada.


### <font color='#2D9CDB'>Q3) Escolha um problema de agrupamento disponível no módulo <a href="https://scikit-learn.org/stable/api/sklearn.datasets.html" target="blank">sklearn.datasets</a> ou no repositório <a href="https://archive.ics.uci.edu/datasets/?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true" target="blank">UCI Machine Learning Repository</a>.</font>
<ul>
    <font color='#2D9CDB'><li>Utilizando um modelo de agrupamento diferente daquele empregado nos exemplos da apostila, realize a otimização de hiperparâmetros com Optuna.</li></font>
    <font color='#2D9CDB'><li>Defina um espaço de busca contendo pelo menos três hiperparâmetros, utilize validação cruzada para avaliação e execute pelo menos 30 trials.</li></font>
    <font color='#2D9CDB'><li>Apresente os melhores hiperparâmetros encontrados, a métrica de classificação utilizada e o desempenho obtido pelo modelo otimizado.</li></font>
    <font color='#2D9CDB'><li>Em seguida, compare os resultados com aqueles obtidos utilizando a configuração padrão do algoritmo e discuta os ganhos observados.</li></font>
</ul>
<font color='2D9CDB'>Não é permitido utilizar o mesmo dataset nem o mesmo modelo apresentado nos exemplos da apostila.</font><br>
<font color='2D9CDB'>Também é possível escolher um conjuntos de dados de classificação ou regressão e descartar a variável-alvo.</font>

In [37]:
dados = load_breast_cancer()

X = dados.data

print("Dataset:", dados.DESCR.split("\n")[0])
print("Quantidade de amostras:", X.shape[0])
print("Número de atributos:", X.shape[1])
print("Variável-alvo descartada para problema de agrupamento")

Dataset: .. _breast_cancer_dataset:
Quantidade de amostras: 569
Número de atributos: 30
Variável-alvo descartada para problema de agrupamento


In [38]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def avaliar_modelo(params):
    scores = []

    for _, val_index in cv.split(X):
        X_fold = X[val_index]

        modelo = Pipeline([
            ("scaler", StandardScaler()),
            ("cluster", AgglomerativeClustering(**params))
        ])

        labels = modelo.fit_predict(X_fold)

        n_clusters = len(set(labels))

        if n_clusters < 2 or n_clusters >= len(X_fold):
            return -1

        score = silhouette_score(X_fold, labels)
        scores.append(score)

    return np.mean(scores)


def objective(trial):
    linkage = trial.suggest_categorical("linkage", ["ward", "complete", "average", "single"])
    n_clusters = trial.suggest_int("n_clusters", 2, 8)
    compute_full_tree = trial.suggest_categorical("compute_full_tree", ["auto", True])

    params = {
        "n_clusters": n_clusters,
        "linkage": linkage,
        "compute_full_tree": compute_full_tree
    }

    if linkage == "ward":
        params["metric"] = "euclidean"
    else:
        params["metric"] = trial.suggest_categorical("metric", ["euclidean", "manhattan", "cosine"])

    return avaliar_modelo(params)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=30)

print("Melhores hiperparâmetros encontrados:")
print(study.best_params)

print(f"\nMelhor Silhouette Score médio: {study.best_value:.4f}")

[I 2026-06-28 21:22:09,550] A new study created in memory with name: no-name-22bf237d-b848-4806-976c-ecf884b2e845
[I 2026-06-28 21:22:09,657] Trial 0 finished with value: 0.18183260867708953 and parameters: {'linkage': 'complete', 'n_clusters': 3, 'compute_full_tree': 'auto', 'metric': 'euclidean'}. Best is trial 0 with value: 0.18183260867708953.
[I 2026-06-28 21:22:09,745] Trial 1 finished with value: 0.18183260867708953 and parameters: {'linkage': 'complete', 'n_clusters': 3, 'compute_full_tree': True, 'metric': 'euclidean'}. Best is trial 0 with value: 0.18183260867708953.
[I 2026-06-28 21:22:09,958] Trial 2 finished with value: 0.04210985311500813 and parameters: {'linkage': 'ward', 'n_clusters': 5, 'compute_full_tree': 'auto'}. Best is trial 0 with value: 0.18183260867708953.
[I 2026-06-28 21:22:10,055] Trial 3 finished with value: -0.26067722032769575 and parameters: {'linkage': 'single', 'n_clusters': 3, 'compute_full_tree': True, 'metric': 'euclidean'}. Best is trial 0 with va

Melhores hiperparâmetros encontrados:
{'linkage': 'ward', 'n_clusters': 2, 'compute_full_tree': 'auto'}

Melhor Silhouette Score médio: 0.4517


In [39]:
params_padrao = {}

silhouette_padrao = avaliar_modelo(params_padrao)
silhouette_otimizado = avaliar_modelo(study.best_params)

ganho_absoluto = silhouette_otimizado - silhouette_padrao
ganho_percentual = (ganho_absoluto / silhouette_padrao) * 100

resultados = pd.DataFrame({
    "Modelo": ["Agglomerative padrão", "Agglomerative otimizado"],
    "Silhouette Score médio": [silhouette_padrao, silhouette_otimizado]
})

display(resultados)

print(f"Silhouette Score com modelo padrão: {silhouette_padrao:.4f}")
print(f"Silhouette Score com modelo otimizado: {silhouette_otimizado:.4f}")
print(f"Ganho absoluto: {ganho_absoluto:.4f}")
print(f"Ganho percentual: {ganho_percentual:.2f}%")

,Modelo,Silhouette Score médio
0,Agglomerative padrão,0.451745
1,Agglomerative otimizado,0.451745


Silhouette Score com modelo padrão: 0.4517
Silhouette Score com modelo otimizado: 0.4517
Ganho absoluto: 0.0000
Ganho percentual: 0.00%


Foi utilizado o dataset Breast Cancer, disponível no sklearn.datasets, descartando a variável-alvo para transformar o problema em agrupamento. O modelo escolhido foi o AgglomerativeClustering, diferente do modelo utilizado no exemplo da apostila.

A otimização foi feita com Optuna, utilizando 30 trials e validação cruzada com 5 folds. A métrica utilizada foi o Silhouette Score, que avalia a separação e a coesão dos grupos, sendo melhor quando apresenta valores maiores. O espaço de busca incluiu os hiperparâmetros linkage, n_clusters, compute_full_tree e metric.

Os melhores hiperparâmetros encontrados foram: linkage = ward, n_clusters = 2 e compute_full_tree = auto. O modelo padrão obteve Silhouette Score médio de 0.4517, e o modelo otimizado também obteve 0.4517.

Assim, não houve ganho de desempenho em relação à configuração padrão. Isso indica que, para esse conjunto de dados e esse espaço de busca, a configuração padrão do algoritmo já apresentou o melhor resultado encontrado pelo Optuna.


# <font color='green'><u><b>Parte 2 - Aprendizado de Máquina Automatizado (AutoML)</b></u></font>

### <font color='#2D9CDB'>Q4) Leia o artigo científico abaixo. Em seguida, responda às questões a seguir, fundamentando suas respostas com evidências apresentadas no artigo:
<ol>
<font color='#2D9CDB'><li>O que é AutoML e quais etapas do pipeline de Machine Learning podem ser automatizadas;</li></font>
<font color='#2D9CDB'><li>Qual a diferença entre Hyperparameter Optimization (HPO) e AutoML;</li></font>
<font color='#2D9CDB'><li>Qual foi o principal objetivo do estudo;</li></font>
<font color='#2D9CDB'><li>Quais ferramentas apresentaram melhor equilíbrio entre desempenho preditivo e custo computacional;</li></font>
<font color='#2D9CDB'><li>Quais são as principais vantagens e limitações das abordagens AutoML discutidas pelos autores.</li></font>
</ol>

<font color='#2D9CDB'><i>Aragão, M.V.C., Afonso, A.G., Ferraz, R.C. et al. A practical evaluation of AutoML tools for binary, multiclass, and multilabel classification. Nature, Scientific Reports, 17682 (2025). DOI: <a href="https://doi.org/10.1038/s41598-025-02149-x" target="blank">10.1038/s41598-025-02149-x</a></i></font>

**1. AutoML:** é a automação do processo de Machine Learning. Pode automatizar pré-processamento, seleção de atributos, escolha do modelo, otimização de hiperparâmetros, treinamento e avaliação.

**2. HPO x AutoML:** HPO otimiza apenas os hiperparâmetros de um modelo. AutoML é mais amplo, pois pode automatizar várias etapas do pipeline inteiro.

**3. Objetivo do estudo:** comparar 16 ferramentas AutoML em problemas de classificação binária, multiclasse e multirrótulo, usando 21 datasets reais.

**4. Melhor equilíbrio:** o AutoGluon apresentou o melhor equilíbrio entre desempenho preditivo e custo computacional.

**5. Vantagens e limitações:** as vantagens são redução do trabalho manual, facilidade de uso e bons resultados com pouca intervenção. As limitações são maior custo computacional em algumas ferramentas, dificuldade em certos tipos de dados e ausência de uma ferramenta que seja melhor em todos os cenários.


### <font color='#2D9CDB'>Q5) Escolha uma ferramenta AutoML avaliada no artigo de Aragão et al. (2025), como AutoGluon, AutoSklearn, PyCaret, FLAML, TPOT, H2O AutoML ou outra ferramenta analisada pelos autores.</font>
<ol>
<font color='#2D9CDB'><li>Utilize essa ferramenta para resolver novamente o problema de regressão ou classificação desenvolvido na Questão 1 ou na Questão 2.</li></font>
<font color='#2D9CDB'><li>Execute o processo completo de AutoML e apresente o modelo vencedor, a métrica obtida, o tempo total de execução e os principais hiperparâmetros da solução final.</li></font>
<font color='#2D9CDB'><li>Compare os resultados obtidos com aqueles alcançados utilizando Optuna na questão correspondente.</li></font>
<font color='#2D9CDB'><li>Discuta as vantagens e desvantagens observadas, a facilidade de utilização da ferramenta, o custo computacional envolvido e se os resultados obtidos justificam o uso de AutoML para o problema analisado.</li></font>
</ol>
</font>

In [40]:
!pip install -q "flaml[automl]"

In [41]:
from sklearn.datasets import load_diabetes

dados_diabetes = load_diabetes()

X_reg = dados_diabetes.data
y_reg = dados_diabetes.target

print(X_reg.shape)
print(y_reg.shape)

(442, 10)
(442,)


In [42]:
import time
from flaml import AutoML

automl = AutoML()

inicio = time.time()

automl.fit(
    X_train=X_reg,
    y_train=y_reg,
    task="regression",
    metric="rmse",
    time_budget=60,
    eval_method="cv",
    n_splits=5,
    seed=42,
    verbose=0
)

tempo_total = time.time() - inicio

print("Modelo vencedor:")
print(automl.best_estimator)

print("\nHiperparâmetros principais:")
print(automl.best_config)

print(f"\nRMSE médio obtido pelo AutoML: {automl.best_loss:.4f}")
print(f"Tempo total de execução: {tempo_total:.2f} segundos")

print("\nModelo final:")
print(automl.model.estimator)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Modelo vencedor:
sgd

Hiperparâmetros principais:
{'penalty': np.str_('l2'), 'alpha': np.float64(7.554093369495018e-06), 'l1_ratio': np.float64(0.14661072110987033), 'epsilon': np.float64(0.03662301713106504), 'learning_rate': np.str_('optimal'), 'eta0': np.float64(0.006105890471129394), 'power_t': np.float64(0.3503816268889408), 'average': False, 'loss': np.str_('huber')}

RMSE médio obtido pelo AutoML: 55.1146
Tempo total de execução: 60.01 segundos

Modelo final:
SGDRegressor(alpha=np.float64(7.554093369495018e-06),
             learning_rate=np.str_('optimal'), loss=np.str_('huber'),
             penalty=np.str_('l2'), tol=0.0001)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [43]:
rmse_optuna = 54.8832
rmse_automl = automl.best_loss

diferenca_absoluta = rmse_optuna - rmse_automl
diferenca_percentual = (diferenca_absoluta / rmse_optuna) * 100

comparacao = pd.DataFrame({
    "Método": ["Optuna + SVR", "FLAML AutoML"],
    "RMSE médio": [rmse_optuna, rmse_automl]
})

display(comparacao)

print(f"RMSE Optuna + SVR: {rmse_optuna:.4f}")
print(f"RMSE FLAML AutoML: {rmse_automl:.4f}")
print(f"Diferença absoluta: {diferenca_absoluta:.4f}")
print(f"Diferença percentual: {diferenca_percentual:.2f}%")

,Método,RMSE médio
0,Optuna + SVR,54.883200
1,FLAML AutoML,55.114632


RMSE Optuna + SVR: 54.8832
RMSE FLAML AutoML: 55.1146
Diferença absoluta: -0.2314
Diferença percentual: -0.42%


Fui utilizada a ferramenta AutoML FLAML para resolver novamente o problema de regressão com o dataset Diabetes. O processo foi executado com validação cruzada de 5 folds, métrica RMSE e tempo limite de execução definido no código.

O FLAML obteve RMSE médio de 55.1537, enquanto o modelo SVR otimizado com Optuna obteve RMSE médio de 54.8832. Como o RMSE é melhor quando apresenta valores menores, o modelo com Optuna teve desempenho levemente superior.

A diferença absoluta foi de -0.2705, equivalente a aproximadamente -0.49%. Isso indica que, neste caso, o AutoML não superou o modelo ajustado manualmente com Optuna, mas obteve um resultado muito próximo.

A principal vantagem do FLAML foi a facilidade de uso, pois ele automatizou a escolha do modelo e dos hiperparâmetros. A desvantagem foi o custo computacional maior, já que a ferramenta testa diferentes configurações durante o tempo definido dessa forma a cedula do notebook demorou mais pra executar. Para esse problema, o uso de AutoML foi útil pela praticidade, mas não trouxe ganho de desempenho em relação ao Optuna.
